# Test LangChain MCP Server using MCP Client SDK

Using the official `mcp` Python package instead of raw HTTP requests.

In [ ]:
# Install the MCP SDK (run once)
!pip install mcp

In [ ]:
import asyncio
import json
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

In [ ]:
MCP_SERVER_URL = "https://docs.langchain.com/mcp"

## Step 1: Connect and Initialize

The SDK handles the full handshake (initialize + notifications/initialized) automatically.

In [ ]:
async def connect_and_get_server_info():
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            # initialize is called automatically by ClientSession
            await session.initialize()
            print("Connected to MCP server!")
            print(f"Server info: {session.server_info}")
            return session.server_info

await connect_and_get_server_info()

## Step 2: List Tools

Get all available tools with their descriptions and input schemas.

In [ ]:
async def list_tools():
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            tools_result = await session.list_tools()
            
            print(f"Found {len(tools_result.tools)} tools:\n")
            for tool in tools_result.tools:
                print(f"Tool: {tool.name}")
                print(f"Description: {tool.description[:200]}...")
                print(f"Input Schema: {json.dumps(tool.inputSchema, indent=2)}")
                print("-" * 60)
            
            return tools_result.tools

tools = await list_tools()

## Step 3: Search the Docs

Call `search_docs_by_lang_chain` to search documentation.

In [ ]:
async def search_docs(query: str):
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            result = await session.call_tool(
                "search_docs_by_lang_chain",
                {"query": query}
            )
            
            print(f"Search results for: '{query}'\n")
            for content in result.content:
                print(content.text)
            
            return result

result = await search_docs("how to create an agent with LangGraph")

## Step 4: Query the Virtual Filesystem

Use `query_docs_filesystem_docs_by_lang_chain` to explore the docs structure,
read pages, search for keywords, and save the complete directory tree to a .txt file.

In [ ]:
async def query_filesystem(command: str):
    """Run a shell command against the virtual docs filesystem."""
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            result = await session.call_tool(
                "query_docs_filesystem_docs_by_lang_chain",
                {"command": command}
            )
            
            print(f"Command: {command}\n")
            for content in result.content:
                print(content.text)
            
            return result

In [ ]:
# 4a. Get the complete directory structure (depth 3)
result = await query_filesystem("tree / -L 3")

In [ ]:
# 4b. Save the complete directory structure to a .txt file
async def save_structure_to_file():
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            # Get full tree structure
            tree_result = await session.call_tool(
                "query_docs_filesystem_docs_by_lang_chain",
                {"command": "tree / -L 4"}
            )
            tree_output = "\n".join(c.text for c in tree_result.content)
            
            # Get top-level listing with details
            ls_result = await session.call_tool(
                "query_docs_filesystem_docs_by_lang_chain",
                {"command": "ls /"}
            )
            ls_output = "\n".join(c.text for c in ls_result.content)
            
            # Build the .txt file content
            content = f"""LangChain Documentation - Virtual Filesystem Structure
=====================================================
Source: https://docs.langchain.com/mcp
Tool:   query_docs_filesystem_docs_by_lang_chain

=====================================================
TOP-LEVEL LISTING (ls /)
=====================================================
{ls_output}

=====================================================
FULL DIRECTORY TREE (tree / -L 4)
=====================================================
{tree_output}
"""
            
            # Write to file
            with open("langchain_docs_structure.txt", "w") as f:
                f.write(content)
            
            print("Saved to langchain_docs_structure.txt")
            print(f"\nTop-level listing:\n{ls_output}")

await save_structure_to_file()

In [ ]:
# 4c. List contents of specific subdirectories
result = await query_filesystem("ls /tutorials/")

In [ ]:
# 4d. List API reference section
result = await query_filesystem("ls /api-reference/")

In [ ]:
# 4e. Read a specific page (first 80 lines)
result = await query_filesystem("head -80 /quickstart.mdx")

In [ ]:
# 4f. Read the agent-auth.mdx file
result = await query_filesystem("cat /agent-auth.mdx")

In [ ]:
# 4g. Search for files mentioning a keyword
result = await query_filesystem('rg -il "retrieval" /')

In [ ]:
# 4h. Search with context (3 lines around each match)
result = await query_filesystem('rg -C 3 "ChatOpenAI" /')

In [ ]:
# 4i. Count total number of documentation pages
result = await query_filesystem('find / -name "*.mdx" | wc -l')

In [ ]:
# 4j. Read multiple pages at once
result = await query_filesystem("head -30 /quickstart.mdx /use-these-docs.mdx")

## Step 5: List Resources

Check what resources (skills, docs) the server exposes.

In [ ]:
async def list_resources():
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            
            resources_result = await session.list_resources()
            
            print(f"Found {len(resources_result.resources)} resources:\n")
            for resource in resources_result.resources:
                print(f"URI: {resource.uri}")
                print(f"Name: {resource.name}")
                print(f"Description: {resource.description}")
                print(f"MIME Type: {resource.mimeType}")
                print("-" * 60)
            
            return resources_result.resources

resources = await list_resources()

---

## Bonus: Reusable helper with persistent session

Keep the connection open for multiple calls without re-initializing each time.

In [ ]:
async def interactive_session():
    """Run multiple tool calls in a single session."""
    async with streamablehttp_client(MCP_SERVER_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("Session established!\n")
            
            # Call 1: Search
            print("=" * 60)
            print("SEARCH: RAG pipeline")
            print("=" * 60)
            result = await session.call_tool(
                "search_docs_by_lang_chain",
                {"query": "RAG pipeline"}
            )
            for content in result.content:
                print(content.text[:500])
            
            print("\n")
            
            # Call 2: Filesystem
            print("=" * 60)
            print("FILESYSTEM: ls /")
            print("=" * 60)
            result = await session.call_tool(
                "query_docs_filesystem_docs_by_lang_chain",
                {"command": "ls /"}
            )
            for content in result.content:
                print(content.text)

await interactive_session()